# Ionosphere


### Importing Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use("seaborn-whitegrid")
%matplotlib inline

In [ ]:
dataset = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data', header = None)

In [ ]:
dataset.head()

### Data Preparation

#### Defining Column Names

In [ ]:
col_names = []
for i in list(range(1, 35)):
    col_names.append('col_' + str(i))
    
col_names.append('target')

In [ ]:
dataset.columns = col_names

In [ ]:
dataset.head()

#### Encoding Target Variable

In [ ]:
dataset['target'] = dataset['target'].apply(lambda x: 1 if x == 'g' else 0)

In [ ]:
dataset.head()

#### Droping Empty Columns 

In [ ]:
dataset.describe()

In [ ]:
dataset.isna().sum().sum()

In [ ]:
dataset['target'].value_counts().plot.bar()

In [ ]:
dataset['col_1'].value_counts().plot.bar()

In [ ]:
dataset['col_2'].value_counts().plot.bar()

In [ ]:
#Column 2 has all 0's. Lets drop it.

dataset.drop(columns = ['col_2'], inplace = True)

In [ ]:
X = dataset.drop(columns = ['target'])
y = dataset['target']

### Model Building

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import accuracy_score, log_loss

#### Base Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [ ]:
log_classifier = LogisticRegression(random_state = 0)
dt_classifier = DecisionTreeClassifier(random_state = 0)

k = list(range(2, 11))

log_train_error = []
log_test_error = []

log_train_acc = []
log_test_acc = []


dt_train_error = []
dt_test_error = []

dt_train_acc = []
dt_test_acc = []

for i in k:
  kfolds = KFold(n_splits = i, shuffle = True, random_state = 42)
  kfolds.get_n_splits(X, y)

  for train_index, test_index in kfolds.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  log_classifier.fit(X_train, y_train)

  log_train_error.append(log_loss(y_true = y_train, y_pred = log_classifier.predict(X_train)))
  log_test_error.append(log_loss(y_true = y_test, y_pred = log_classifier.predict(X_test)))

  log_train_acc.append(accuracy_score(y_true = y_train, y_pred = log_classifier.predict(X_train)))
  log_test_acc.append(accuracy_score(y_true = y_test, y_pred = log_classifier.predict(X_test)))


  dt_classifier.fit(X_train, y_train)

  dt_train_error.append(log_loss(y_true = y_train, y_pred = dt_classifier.predict(X_train)))
  dt_test_error.append(log_loss(y_true = y_test, y_pred = dt_classifier.predict(X_test)))

  dt_train_acc.append(accuracy_score(y_true = y_train, y_pred = dt_classifier.predict(X_train)))
  dt_test_acc.append(accuracy_score(y_true = y_test, y_pred = dt_classifier.predict(X_test)))

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(k, log_train_error, label = "Logistic Regression (Train) Error")
plt.plot(k, log_test_error, label = "Logistic Regression (Test) Error")
plt.legend()
plt.xlabel("k")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Logistic Regression Loss")

plt.subplot(1, 2, 2)
plt.plot(k, dt_train_error, label = "Decision Tree (Train) Error")
plt.plot(k, dt_test_error, label = "Decision Tree (Test) Error")
plt.legend()
plt.xlabel("k")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Decision Tree Loss")

plt.show()

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(k, log_train_acc, label = "Logistic Regression (Train) Accuracy")
plt.plot(k, log_test_acc, label = "Logistic Regression (Test) Accuracy")
plt.legend()
plt.xlabel("k")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.01])
plt.title("Logistic Regression Accuracy")

plt.subplot(1, 2, 2)
plt.plot(k, dt_train_acc, label = "Decision Tree (Train) Accuracy")
plt.plot(k, dt_test_acc, label = "Decision Tree (Test) Accuracy")
plt.legend()
plt.xlabel("k")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.01])
plt.title("Decision Tree Accuracy")

plt.show()

#### Bagging Classifier

##### Bagging with Varying K and fixed ensembles

In [ ]:
from sklearn.ensemble import BaggingClassifier 

In [ ]:
bag_log_classifier = BaggingClassifier(base_estimator = log_classifier, n_estimators = 100, random_state = 0)
bag_dt_classifier = BaggingClassifier(base_estimator = dt_classifier, n_estimators = 100, random_state = 0)

k = list(range(2, 11))

bag_log_train_acc = []
bag_log_test_acc = []

bag_log_train_error = []
bag_log_test_error = []


bag_dt_train_acc = []
bag_dt_test_acc = []

bag_dt_train_error = []
bag_dt_test_error = []

for i in k:
  kfolds = KFold(n_splits = i, shuffle = True, random_state = 42)
  kfolds.get_n_splits(X, y)

  for train_index, test_index in kfolds.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  bag_log_classifier.fit(X_train, y_train)

  bag_log_train_error.append(log_loss(y_true = y_train, y_pred = bag_log_classifier.predict(X_train)))
  bag_log_test_error.append(log_loss(y_true = y_test, y_pred = bag_log_classifier.predict(X_test)))

  bag_log_train_acc.append(accuracy_score(y_true = y_train, y_pred = bag_log_classifier.predict(X_train)))
  bag_log_test_acc.append(accuracy_score(y_true = y_test, y_pred = bag_log_classifier.predict(X_test)))


  bag_dt_classifier.fit(X_train, y_train)
  
  bag_dt_train_error.append(log_loss(y_true = y_train, y_pred = bag_dt_classifier.predict(X_train)))
  bag_dt_test_error.append(log_loss(y_true = y_test, y_pred = bag_dt_classifier.predict(X_test)))

  bag_dt_train_acc.append(accuracy_score(y_true = y_train, y_pred = bag_dt_classifier.predict(X_train)))
  bag_dt_test_acc.append(accuracy_score(y_true = y_test, y_pred = bag_dt_classifier.predict(X_test)))

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(k, bag_log_train_error, label = "Bagging - Logistic Regression (Train) Error")
plt.plot(k, bag_log_test_error, label = "Bagging - Logistic Regression (Test) Error")
plt.legend()
plt.xlabel("k")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Bagging - Logistic Regression Loss")

plt.subplot(1, 2, 2)
plt.plot(k, bag_dt_train_error, label = "Bagging - Decision Tree (Train) Error")
plt.plot(k, bag_dt_test_error, label = "Bagging - Decision Tree (Test) Error")
plt.legend()
plt.xlabel("k")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Bagging - Decision Tree Loss")

plt.show()

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(k, bag_log_train_acc, label = "Bagging - Logistic Regression (Train) Accuracy")
plt.plot(k, bag_log_test_acc, label = "Bagging - Logistic Regression (Test) Accuracy")
plt.legend()
plt.xlabel("k")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.01])
plt.title("Bagging - Logistic Regression")

plt.subplot(1, 2, 2)
plt.plot(k, bag_dt_train_acc, label = "Bagging - Decision Tree (Train) Accuracy")
plt.plot(k, bag_dt_test_acc, label = "Bagging - Decision Tree (Test) Accuracy")
plt.legend()
plt.xlabel("k")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.01])
plt.title("Bagging - Decision Tree")

plt.show()

##### Bagging with fixed K and varying ensembles

In [ ]:
k = 7

bag_log_train_acc = []
bag_log_test_acc = []

bag_log_train_error = []
bag_log_test_error = []


bag_dt_train_acc = []
bag_dt_test_acc = []

bag_dt_train_error = []
bag_dt_test_error = []

kfolds = KFold(n_splits = i, shuffle = True, random_state = 42)
kfolds.get_n_splits(X, y)

for train_index, test_index in kfolds.split(X, y):
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

ensemb = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

for e in ensemb:

  bag_log_classifier = BaggingClassifier(base_estimator = log_classifier, n_estimators = e, random_state = 0)
  bag_dt_classifier = BaggingClassifier(base_estimator = dt_classifier, n_estimators = e, random_state = 0)

  bag_log_classifier.fit(X_train, y_train)

  bag_log_train_error.append(log_loss(y_true = y_train, y_pred = bag_log_classifier.predict(X_train)))
  bag_log_test_error.append(log_loss(y_true = y_test, y_pred = bag_log_classifier.predict(X_test)))

  bag_log_train_acc.append(accuracy_score(y_true = y_train, y_pred = bag_log_classifier.predict(X_train)))
  bag_log_test_acc.append(accuracy_score(y_true = y_test, y_pred = bag_log_classifier.predict(X_test)))


  bag_dt_classifier.fit(X_train, y_train)
  
  bag_dt_train_error.append(log_loss(y_true = y_train, y_pred = bag_dt_classifier.predict(X_train)))
  bag_dt_test_error.append(log_loss(y_true = y_test, y_pred = bag_dt_classifier.predict(X_test)))

  bag_dt_train_acc.append(accuracy_score(y_true = y_train, y_pred = bag_dt_classifier.predict(X_train)))
  bag_dt_test_acc.append(accuracy_score(y_true = y_test, y_pred = bag_dt_classifier.predict(X_test)))

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(ensemb, bag_log_train_error, label = "Bagging - Logistic Regression (Train) Error")
plt.plot(ensemb, bag_log_test_error, label = "Bagging - Logistic Regression (Test) Error")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Bagging - Logistic Regression Loss")

plt.subplot(1, 2, 2)
plt.plot(ensemb, bag_dt_train_error, label = "Bagging - Decision Tree (Train) Error")
plt.plot(ensemb, bag_dt_test_error, label = "Bagging - Decision Tree (Test) Error")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Bagging - Decision Tree Loss")

plt.show()

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(ensemb, bag_log_train_acc, label = "Bagging - Logistic Regression (Train) Accuracy")
plt.plot(ensemb, bag_log_test_acc, label = "Bagging - Logistic Regression (Test) Accuracy")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.1])
plt.title("Bagging - Logistic Regression")

plt.subplot(1, 2, 2)
plt.plot(ensemb, bag_dt_train_acc, label = "Bagging - Decision Tree (Train) Accuracy")
plt.plot(ensemb, bag_dt_test_acc, label = "Bagging - Decision Tree (Test) Accuracy")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.1])
plt.title("Bagging - Decision Tree")

plt.show()

#### AdaBoost Classifier

##### Boosting with Varying K and fixed ensembles

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

In [ ]:
boost_log_classifier = AdaBoostClassifier(base_estimator = log_classifier, n_estimators = 100, random_state = 0)
boost_dt_classifier = AdaBoostClassifier(base_estimator = dt_classifier, n_estimators = 100, random_state = 0)

k = list(range(2, 11))

boost_log_train_acc = []
boost_log_test_acc = []

boost_log_train_error = []
boost_log_test_error = []


boost_dt_train_acc = []
boost_dt_test_acc = []

boost_dt_train_error = []
boost_dt_test_error = []


for i in k:
  kfolds = KFold(n_splits = i, shuffle = True, random_state = 42)
  kfolds.get_n_splits(X, y)

  for train_index, test_index in kfolds.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  boost_log_classifier.fit(X_train, y_train)

  boost_log_train_error.append(log_loss(y_true = y_train, y_pred = boost_log_classifier.predict(X_train)))
  boost_log_test_error.append(log_loss(y_true = y_test, y_pred = boost_log_classifier.predict(X_test)))

  boost_log_train_acc.append(accuracy_score(y_true = y_train, y_pred = boost_log_classifier.predict(X_train)))
  boost_log_test_acc.append(accuracy_score(y_true = y_test, y_pred = boost_log_classifier.predict(X_test)))


  boost_dt_classifier.fit(X_train, y_train)

  boost_dt_train_error.append(log_loss(y_true = y_train, y_pred = boost_dt_classifier.predict(X_train)))
  boost_dt_test_error.append(log_loss(y_true = y_test, y_pred = boost_dt_classifier.predict(X_test)))

  boost_dt_train_acc.append(accuracy_score(y_true = y_train, y_pred = boost_dt_classifier.predict(X_train)))
  boost_dt_test_acc.append(accuracy_score(y_true = y_test, y_pred = boost_dt_classifier.predict(X_test)))

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(k, boost_log_train_error, label = "Boosting - Logistic Regression (Train) Error")
plt.plot(k, boost_log_test_error, label = "Boosting - Logistic Regression (Test) Error")
plt.legend()
plt.xlabel("k")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Boosting - Logistic Regression Loss")

plt.subplot(1, 2, 2)
plt.plot(k, boost_dt_train_error, label = "Boosting - Decision Tree (Train) Error")
plt.plot(k, boost_dt_test_error, label = "Boosting - Decision Tree (Test) Error")
plt.legend()
plt.xlabel("k")
plt.ylim([-1, 30])
plt.ylabel("Error")
plt.title("Boosting - Decision Tree Loss")

plt.show()

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(k, boost_log_train_acc, label = "Boosting - Logistic Regression (Train) Accuracy")
plt.plot(k, boost_log_test_acc, label = "Boosting - Logistic Regression (Test) Accuracy")
plt.legend()
plt.xlabel("k")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.01])
plt.title("Boosting - Logistic Regression")

plt.subplot(1, 2, 2)
plt.plot(k, boost_dt_train_acc, label = "Boosting - Decision Tree (Train) Accuracy")
plt.plot(k, boost_dt_test_acc, label = "Boosting - Decision Tree (Test) Accuracy")
plt.legend()
plt.xlabel("k")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.01])
plt.title("Boosting - Decision Tree")

plt.show()

##### Boosting with fixed K and varying ensembles

In [ ]:
k = 7

boost_log_train_acc = []
boost_log_test_acc = []

boost_log_train_error = []
boost_log_test_error = []


boost_dt_train_acc = []
boost_dt_test_acc = []

boost_dt_train_error = []
boost_dt_test_error = []

kfolds = KFold(n_splits = k, shuffle = True, random_state = 42)
kfolds.get_n_splits(X, y)

for train_index, test_index in kfolds.split(X, y):
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  
ensemb = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

for e in ensemb:

  boost_log_classifier = AdaBoostClassifier(base_estimator = log_classifier, n_estimators = e, random_state = 42)
  boost_dt_classifier = AdaBoostClassifier(base_estimator = dt_classifier, n_estimators = e, random_state = 42)

  boost_log_classifier.fit(X_train, y_train)

  boost_log_train_error.append(log_loss(y_true = y_train, y_pred = boost_log_classifier.predict(X_train)))
  boost_log_test_error.append(log_loss(y_true = y_test, y_pred = boost_log_classifier.predict(X_test)))

  boost_log_train_acc.append(accuracy_score(y_true = y_train, y_pred = boost_log_classifier.predict(X_train)))
  boost_log_test_acc.append(accuracy_score(y_true = y_test, y_pred = boost_log_classifier.predict(X_test)))


  boost_dt_classifier.fit(X_train, y_train)

  boost_dt_train_error.append(log_loss(y_true = y_train, y_pred = boost_dt_classifier.predict(X_train)))
  boost_dt_test_error.append(log_loss(y_true = y_test, y_pred = boost_dt_classifier.predict(X_test)))

  boost_dt_train_acc.append(accuracy_score(y_true = y_train, y_pred = boost_dt_classifier.predict(X_train)))
  boost_dt_test_acc.append(accuracy_score(y_true = y_test, y_pred = boost_dt_classifier.predict(X_test)))

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(ensemb, boost_log_train_error, label = "Boosting - Logistic Regression (Train) Error")
plt.plot(ensemb, boost_log_test_error, label = "Boosting - Logistic Regression (Test) Error")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Error")
plt.ylim([-1, 30])
plt.title("Boosting - Logistic Regression Loss")

plt.subplot(1, 2, 2)
plt.plot(ensemb, boost_dt_train_error, label = "Boosting - Decision Tree (Train) Error")
plt.plot(ensemb, boost_dt_test_error, label = "Boosting - Decision Tree (Test) Error")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylim([-1, 30])
plt.ylabel("Error")
plt.title("Boosting - Decision Tree Loss")

plt.show()

In [ ]:
plt.figure(figsize=(20, 6))

plt.subplot(1, 2, 1)
plt.plot(ensemb, boost_log_train_acc, label = "Logistic Regression (Train) Accuracy")
plt.plot(ensemb, boost_log_test_acc, label = "Logistic Regression (Test) Accuracy")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.1])
plt.title("Logistic Regression for num_estimators = {}".format(e))

plt.subplot(1, 2, 2)
plt.plot(ensemb, boost_dt_train_acc, label = "Decission Tree (Train) Accuracy")
plt.plot(ensemb, boost_dt_test_acc, label = "Decission Tree (Test) Accuracy")
plt.legend()
plt.xlabel("Number of Ensembles")
plt.ylabel("Accuracy %")
plt.ylim([0, 1.1])
plt.title("Decission Tree for num_estimators = {}".format(e))

plt.show()